## Setup

In [1]:
import os
import re
import json
from typing import List

from openai import OpenAI
from nemo_curator import OpenAIClient
from nemo_curator.synthetic import NemotronGenerator

In [2]:
openai_client = OpenAI(
    base_url=os.environ["NVIDIA_BASE_URL"],
    api_key=os.environ["NVIDIA_API_KEY"],
)

curator_client = OpenAIClient(openai_client)
generator = NemotronGenerator(curator_client)

MODEL = "mistralai/mistral-7b-instruct-v0.3"

MODEL_KWARGS = {
    "temperature": 0.3,
    "top_p": 0.9,
    "max_tokens": 512,
}

## robust parsing & filtering

In [3]:
def parse_numbered_list(text: str) -> List[str]:
    items = []
    for line in text.splitlines():
        m = re.match(r"\s*\d+\.\s*(.+)", line)
        if m:
            item = m.group(1).strip()
            item = item.split(":")[0].strip()
            items.append(item)
    return items


BAD_PATTERNS = [
    "i assume",
    "i'll assume",
    "assuming that",
    "as i understand it",
]

def is_bad_answer(answer: str) -> bool:
    return any(p in answer.lower() for p in BAD_PATTERNS)

## Generate macro topics

In [4]:
N_MACRO_TOPICS = 20

raw_macro = generator.generate_macro_topics(
    model=MODEL,
    model_kwargs=MODEL_KWARGS,
    n_macro_topics=N_MACRO_TOPICS,
)

macro_topics = parse_numbered_list(raw_macro[0])

print("Macro topics:")
for t in macro_topics:
    print("-", t)

print("Count:", len(macro_topics))

Macro topics:
- Food and Nutrition
- Technology
- Climate Change
- Renewable Energy
- Mental Health
- Global Politics
- Space Exploration
- Wildlife Conservation
- Architecture
- Literature
- Music
- Art
- Philosophy
- Economics
- History
- Education
- Healthcare
- Travel
- Social Issues
- Science and Technology Ethics
Count: 20


##  Generate grounded subtopics

In [5]:
SUBTOPIC_PROMPT = (
    "Generate {n_subtopics} concrete, specific subtopics of the macro topic: {macro_topic}.\n"
    "Each subtopic must be meaningful and self-contained.\n"
    "Return ONLY a numbered list.\n"
)

def generate_subtopics(macro_topic: str, n_subtopics: int = 4) -> List[str]:
    raw = generator.generate_subtopics(
        model=MODEL,
        model_kwargs=MODEL_KWARGS,
        macro_topic=macro_topic,
        n_subtopics=n_subtopics,
        prompt_template=SUBTOPIC_PROMPT,
    )
    subs = parse_numbered_list(raw[0])
    return [s for s in subs if "topic" not in s.lower()]

## Collect ALL subtopics

In [6]:
all_subtopics = []

for macro in macro_topics:
    subs = generate_subtopics(macro, n_subtopics=4)
    all_subtopics.extend(subs)

print("Total subtopics:", len(all_subtopics))

Total subtopics: 86


## Generate questions

In [7]:
def generate_questions(topic: str, n_questions: int = 3) -> List[str]:
    prompt = (
        f"Generate {n_questions} clear, diverse questions about:\n"
        f"{topic}\n"
        "Return ONLY a numbered list."
    )

    raw = generator.generate_open_qa_from_topic(
        model=MODEL,
        model_kwargs=MODEL_KWARGS,
        topic=topic,
        n_openlines=n_questions,
        prompt_template=prompt,
    )

    return parse_numbered_list(raw[0])

## Generate answers

In [8]:
def generate_answer(question: str) -> str:
    dialogue = generator.generate_dialogue(
        openline=question,
        user_model=MODEL,
        user_model_kwargs=MODEL_KWARGS,
        assistant_model=MODEL,
        assistant_model_kwargs=MODEL_KWARGS,
        n_user_turns=1,
    )
    return dialogue[1]["content"].strip()

## Build raw SFT samples

In [11]:
raw_samples = []

for subtopic in all_subtopics:
    questions = generate_questions(subtopic, n_questions=3)
   
    for q in questions:
        a = generate_answer(q)
        if not is_bad_answer(a):
            raw_samples.append({
                "topic": subtopic,
                "question": q,
                "answer": a,
            })

print("Raw samples:", len(raw_samples))
with open("raw_samples.txt", "w", encoding="utf-8") as f:
    for i, s in enumerate(raw_samples, start=1):
        f.write(f"=== SAMPLE {i} ===\n")
        f.write(f"TOPIC: {s['topic']}\n\n")
        f.write(f"Q: {s['question']}\n\n")
        f.write(f"A:\n{s['answer']}\n\n")
        f.write("=" * 50 + "\n\n")

print("Saved raw_samples.txt")
def compose_messages(q, a):
    return [
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ]


def get_reward(messages):
    r = openai_client.chat.completions.create(
        model="nvidia/llama-3.1-nemotron-70b-reward",
        messages=messages,
    )
    return float(r.choices[0].message.content.split(":")[-1])


MIN_REWARD = -5.0

filtered = []

for s in raw_samples:
    reward = get_reward(compose_messages(s["question"], s["answer"]))
    if reward >= MIN_REWARD:
        s["reward"] = reward
        filtered.append(s)

print("Filtered samples:", len(filtered))


Raw samples: 258
Saved raw_samples.txt
Filtered samples: 18


## Export clean JSONL

In [10]:
with open("sft_dataset.jsonl", "w") as f:
    for s in filtered:
        row = {
            "messages": [
                {"role": "user", "content": s["question"]},
                {"role": "assistant", "content": s["answer"]},
            ],
            "metadata": {
                "topic": s["topic"],
                "reward": s["reward"],
                "source": "synthetic-nemo-curator",
            }
        }
        f.write(json.dumps(row) + "\n")

print("Saved sft_dataset.jsonl")

Saved sft_dataset.jsonl
